# 第10章 性能評価

## 10.3 LLMを用いた自動評価

### 10.3.2 Japanese Vicuna QA Benchmarkによる自動評価

#### 環境の準備

In [1]:
!pip install bitsandbytes 'datasets<4.0.0' transformers[torch,sentencepiece] openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 34.7 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
from transformers.trainer_utils import set_seed
set_seed(42)

#### データセットの準備

In [3]:
from datasets import load_dataset

test_dataset = load_dataset(
    "llm-book/ja-vicuna-qa-benchmark", split="test"
)
print(test_dataset)

README.md:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/10.1k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/80 [00:00<?, ? examples/s]

Dataset({
    features: ['question_id', 'category', 'turns'],
    num_rows: 80
})


In [4]:
# データを表示
test_data = test_dataset[0]
print(test_dataset[0])

{'question_id': 1, 'category': 'generic', 'turns': ['時間管理能力を向上させるにはどうしたらいいですか？']}


#### パイプラインの作成

In [5]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

model_name = "tokyotech-llm/Swallow-7b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=False,
    device_map="auto",
)

config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/914k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

In [6]:
from transformers import pipeline

generation_config = {
    "do_sample": True,
    "max_new_tokens": 2048,
    "temperature": 0.99,
    "top_p": 0.95
}
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    **generation_config
)

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'top_p', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


#### 質問に回答するためのプロンプトの作成

In [7]:
prompt_template = """
以下に、あるタスクを説明する指示があります。
リクエストを適切に完了するための回答を記述してください。\n\n### 指示:\n{instruction}\n\n### 応答:\n
"""
# プロンプトテンプレートの{instruction}に入力テキストに置換する
prompt = prompt_template.format(instruction=test_data["turns"][0])
print(prompt)


以下に、あるタスクを説明する指示があります。
リクエストを適切に完了するための回答を記述してください。

### 指示:
時間管理能力を向上させるにはどうしたらいいですか？

### 応答:




#### 評価対象LLMによる質問の回答生成

In [9]:
# 質問の回答を生成する
output = text_generation_pipeline(prompt)
# プロンプト部分を削除して回答のみにする
generated_text_swallow = output[0]["generated_text"].replace(
    prompt, ""
)
print(generated_text_swallow)

Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


 時間 管理 能力 を 向上 させる 方法 に は 、 以下 の 方法 が 考え られ ます 。 <0x0A> <0x0A> 1. ▁** 目標 設定 ** : ▁ 時間 を 割 い て いる タスク ごと に 目標 を 設定 し 、 達成 する こと を 意識 して 実行 し ます 。 <0x0A> 2. ▁** 時間 の 追跡 ** : ▁ 時間 を 記録 する ため の タイム トラ ッキング アプリ や 、 タスク 管理 ソフトウェア を 使用 し 、 時間 の 使い 方 を 可視 化 し ます 。 <0x0A> 3. ▁** 時間 の 優先 順位 付け ** : ▁ タスク を 重要 度 や 緊急 度 に 基づい て 優先 順位 を つけ 、 優先 度 の 高い タスク から 開始 し 、 時間 を 効率 的 に 利用 し ます 。 <0x0A> 4. ▁** 時間 の 割り 当て ** : ▁ タスク を 分割 し 、 それ に 時間 を 割り 当て 、 タスク を 1 つ ずつ 完了 して いく こと で 、 タスク の 進み 具合 を 確認 し ながら 時間 を 管理 し ます 。 <0x0A> 5. ▁** 時間 の 計画 ** : ▁ 時間 を 計画 的 に 計画 し 、 タスク を 予定 し 、 それ に 基づい て 時間 を 割り 当て ます 。 <0x0A> 6. ▁** 時間 の リ マ イン ダー ** : ▁ 時間 リ マ イン ダー を 設定 して 、 時間 を 忘れ ず に タスク を 完了 する ため に 、 時間 を 把握 して 時間 の 管理 を 促進 し ます 。 <0x0A> 7. ▁** 時間 の 訓練 ** : ▁ 時間 管理 の 能力 を 向上 させる ため に 、 時間 管理 トレーニング や ワークショップ に 参加 する こと も 重要 です 。 <0x0A> <0x0A> 上記 の 方法 を 適用 する こと で 、 時間 管理 能力 を 向上 させ 、 生産 性 と 効率 性 を 上げる こと が でき ます 。


#### 評価者LLMによる単一採点の実施

In [10]:
from pprint import pprint

# 単一採点のためのプロンプトを作成する
single_judge_prompt_template = """
[インストラクション]
\n以下に示されるユーザの質問に対してAIアシスタントが提供した回答の質を評価してください。具体的には、回答の有用性、関連性、正確性、深さ、創造性、詳細レベルなどの要素を考慮して評価してください。評価の際には、まず回答内容を簡単に、できるだけ客観的に説明してください。説明を行った後、必ず「[[rating]]」という形式で、回答を1から10の尺度で評価してください（例：[[5]]）。\".
\n
\n[ユーザの質問]\n{question}
\n
\n[アシスタントの答えの始まり]\n{answer}\n[アシスタントの答えの終わり]
"""

# OpenAI API に渡す入力を作成する
messages = [
    {
        "role": "system",
        "content": "あなたは役に立つアシスタントです",
    },
    {
        "role": "user",
        "content": single_judge_prompt_template.format(
            question=test_data["turns"][0],
            answer=generated_text_swallow
        )
    }
]
pprint(messages)

[{'content': 'あなたは役に立つアシスタントです', 'role': 'system'},
 {'content': '\n'
             '[インストラクション]\n'
             '\n'
             '以下に示されるユーザの質問に対してAIアシスタントが提供した回答の質を評価してください。具体的には、回答の有用性、関連性、正確性、深さ、創造性、詳細レベルなどの要素を考慮して評価してください。評価の際には、まず回答内容を簡単に、できるだけ客観的に説明してください。説明を行った後、必ず「[[rating]]」という形式で、回答を1から10の尺度で評価してください（例：[[5]]）。".\n'
             '\n'
             '\n'
             '\n'
             '[ユーザの質問]\n'
             '時間管理能力を向上させるにはどうしたらいいですか？\n'
             '\n'
             '\n'
             '\n'
             '[アシスタントの答えの始まり]\n'
             ' 時間 管理 能力 を 向上 させる 方法 に は 、 以下 の 方法 が 考え られ ます 。 <0x0A> <0x0A> '
             '1. ▁** 目標 設定 ** : ▁ 時間 を 割 い て いる タスク ごと に 目標 を 設定 し 、 達成 する こと '
             'を 意識 して 実行 し ます 。 <0x0A> 2. ▁** 時間 の 追跡 ** : ▁ 時間 を 記録 する ため の '
             'タイム トラ ッキング アプリ や 、 タスク 管理 ソフトウェア を 使用 し 、 時間 の 使い 方 を 可視 化 し ます '
             '。 <0x0A> 3. ▁** 時間 の 優先 順位 付け ** : ▁ タスク を 重要 度 や 緊急 度 に 基づい て '
             '優先 順位 を つけ 、 優先 度 の 高い タスク から 開始 し 、 時間 を 効率 的 に 利

In [14]:
from openai import OpenAI
from google.colab import userdata

# 評価者LLMとしてGPT-4を用いて、単一採点による評価を行う
# リクエストのパラメータを準備
params = {
    "messages": messages,
    "max_tokens": 2048,
    "model": "gpt-4-turbo-2024-04-09"
}
# OpenAI APIのクライアントを初期化
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)
# OpenAI APIにリクエストを送信
response = client.chat.completions.create(**params)
# レスポンスからLLMの応答を取得
content = response.choices[0].message.content
print(content)

アシスタントの回答は、時間管理能力を向上させるための様々な戦略を提供しています。これには目標設定、時間の追跡、優先順位付け、タスクの時間割り当て、計画作成、リマインダーの設定、および時間管理に関するトレーニングやワークショップへの参加が含まれています。

回答は具体的なアプローチとツールを提供しており、時間管理を改善するために実際に取り入れることができる手段を詳しく説明しています。各ステップは明確で具体的であり、読者が実際にアクションを起こしやすいように設計されています。

ただし、回答は基本的な時間管理手法に焦点を当てており、より発展的なテクニックや心理的な側面に触れていないため、ある程度時間管理に精通しているユーザーには新鮮さが少なく感じられるかもしれません。

評価としては、回答が基礎的であるものの、提供するアドバイスは実際的でアクセスしやすく、広範囲の読者に適用可能であるため高い評価をします。[[8]]


#### GPT4.1-miniによる質問の回答の生成

In [15]:
# GPT-3.5で質問の回答を生成する
messages = [
    {
        "role": "system",
        "content": "あなたは役に立つアシスタントです。"
    },
    {
        "role": "user",
        "content": prompt
    }
]
# リクエストのパラメータを準備
params = {
    "messages": messages,
    "max_tokens": 2048,
    "model": "gpt-4.1-mini-2025-04-14"
}
# OpenAI APIにリクエストを送信
response = client.chat.completions.create(**params)
# レスポンスからLLMの応答を取得
generated_text_gpt_4_1_mini = response.choices[0].message.content
print(generated_text_gpt_4_1_mini)

時間管理能力を向上させるためには、以下の方法を試してみてください。

1. **目標を明確にする**  
   何を達成したいのか具体的な目標を設定し、それに向けた優先順位を決めます。

2. **計画を立てる**  
   毎日のスケジュールや週間予定を作成し、やるべきことを細かく書き出しましょう。ToDoリストやカレンダーを活用すると効果的です。

3. **優先順位をつける**  
   緊急度と重要度に基づいてタスクを分類し、重要かつ緊急なものから取り組みます。例えば「Eisenhowerマトリックス」を参考にすると良いでしょう。

4. **時間を区切る**  
   ポモドーロ・テクニック（25分作業＋5分休憩）など時間を区切って作業する方法を試すことで、集中力が高まります。

5. **集中力を高める環境を作る**  
   作業に集中できる場所を整え、スマホの通知をオフにするなど、気が散る要素を減らしましょう。

6. **一度に一つのことに集中する**  
   マルチタスクを避け、ひとつの作業に集中して効率よく終わらせることを心がけます。

7. **進捗を振り返る**  
   定期的に自分の時間の使い方や達成度を見直し、改善点を見つけて修正していきましょう。

8. **無理をしない**  
   休憩やリフレッシュの時間も計画に入れ、バランスよく過ごすことが継続の鍵です。

これらの方法を組み合わせて実践することで、徐々に時間管理のスキルが向上していきます。まずはできることから始めてみてください。


#### ペア比較のためのプロンプトの作成

In [16]:
# ペア比較のためのプロンプトを作成する
pair_judge_prompt_template = """
[ユーザーの質問]
\n{question}
\n
\n[アシスタントAの答えの始まり]
\n{answer_a}
\n[アシスタントAの答えの終わり]
\n
\n[アシスタントBの答えの始まり]
\n{answer_b}
\n[アシスタントBの答えの終わり]
"""

# OpenAI APIに渡す入力を作成
messages = [
    {
        "role": "system",
        "content": (
            "以下に示されるユーザの質問に対して2人のAIアシスタントが提供した回答の質を評価してください。"
            "回答の内容がユーザの指示に従っており、"
            "ユーザの質問によりよく答えているアシスタントを選んでください。"
            "具体的には、回答の有用性、関連性、正確性、深さ、創造性、"
            "詳細レベルなどの要素を考慮する必要があります。"
            "評価の際には、まず2つの回答を比較し、"
            "簡単な説明をしてください。立場が偏らないようにし、"
            "回答の提示順があなたの判断に影響しないようにしてください。"
            "回答の長さが評価に影響しないこと、"
            "特定のアシスタントの名前を好まないこと、"
            "できるだけ客観的であること、に気をつけてください。"
            "説明の後に、"
            "最終的な判断を以下の形式に従って出力してください：アシスタントAが優れていれば[[A]]、"
            "アシスタントBが優れていれば[[B]]、同点の場合は[[C]]"
        ),
    },
    {
        "role": "user",
        "content": pair_judge_prompt_template.format(
            question=test_data["turns"][0],
            answer_a=generated_text_swallow,
            answer_b=generated_text_gpt_4_1_mini,
        ),
    },
]
pprint(messages)

[{'content': '以下に示されるユーザの質問に対して2人のAIアシスタントが提供した回答の質を評価してください。回答の内容がユーザの指示に従っており、ユーザの質問によりよく答えているアシスタントを選んでください。具体的には、回答の有用性、関連性、正確性、深さ、創造性、詳細レベルなどの要素を考慮する必要があります。評価の際には、まず2つの回答を比較し、簡単な説明をしてください。立場が偏らないようにし、回答の提示順があなたの判断に影響しないようにしてください。回答の長さが評価に影響しないこと、特定のアシスタントの名前を好まないこと、できるだけ客観的であること、に気をつけてください。説明の後に、最終的な判断を以下の形式に従って出力してください：アシスタントAが優れていれば[[A]]、アシスタントBが優れていれば[[B]]、同点の場合は[[C]]',
  'role': 'system'},
 {'content': '\n'
             '[ユーザーの質問]\n'
             '\n'
             '時間管理能力を向上させるにはどうしたらいいですか？\n'
             '\n'
             '\n'
             '\n'
             '[アシスタントAの答えの始まり]\n'
             '\n'
             ' 時間 管理 能力 を 向上 させる 方法 に は 、 以下 の 方法 が 考え られ ます 。 <0x0A> <0x0A> '
             '1. ▁** 目標 設定 ** : ▁ 時間 を 割 い て いる タスク ごと に 目標 を 設定 し 、 達成 する こと '
             'を 意識 して 実行 し ます 。 <0x0A> 2. ▁** 時間 の 追跡 ** : ▁ 時間 を 記録 する ため の '
             'タイム トラ ッキング アプリ や 、 タスク 管理 ソフトウェア を 使用 し 、 時間 の 使い 方 を 可視 化 し ます '
             '。 <0x0A> 3. ▁** 時間 の 優先 順位 付け ** : ▁ タスク を 重要 度 や 緊急

#### 評価者LLMによるペア比較の実施

In [17]:
params = {
    "messages": messages,
    "max_tokens": 2048,
    "model": "gpt-4.1-mini-2025-04-14",
}
# OpenAI APIにリクエストを送信
response = client.chat.completions.create(**params)
# レスポンスからLLMの応答を取得
content = response.choices[0].message.content
print(content)

両者の回答は時間管理能力の向上に関する実用的なアドバイスを提供していますが、差異もあります。

アシスタントAは箇条書きで具体的な項目を列挙し、ツールやトレーニング参加の提案まで含めており全体的な枠組みを提示しています。ただし、文章中に不自然な文字列（例えば「<0x0A>」や「▁」）が混入しており、読みやすさや直感的理解が妨げられています。また、内容がやや重複気味であり、ポイントの説明も簡潔すぎて深みが乏しい印象です。

一方アシスタントBは、具体的なテクニック（Eisenhowerマトリックス、ポモドーロ・テクニック）や集中力を高めるための環境づくり、マルチタスク回避、休憩も含めたバランスの重要性など、より実践的かつ多角的なアプローチを丁寧に解説しています。文章も読みやすく整理されており、初心者にも分かりやすい構成です。

以上を踏まえると、ユーザが「時間管理能力を向上させるにはどうしたらいいか」という質問に対しては、具体的かつ実行しやすい方法を分かりやすく提示しているアシスタントBの回答の方が優れていると評価できます。

よって、最終判断は [[B]] です。
